# core

> Lightweight extensions to python-docx: markdown rendering, markdown-to-runs parsing, and tracked changes (insertions/deletions with author attribution).

In [ ]:
#| default_exp core

In [ ]:
#| export
from copy import deepcopy
from datetime import datetime, timezone
from docx import Document as DocxDocument
from docx.document import Document
from docx.text.run import Run
from docx.text.paragraph import Paragraph
from docx.table import Table
from docx.oxml import OxmlElement
from docx.oxml.table import CT_Tbl
from docx.oxml.ns import qn
from docx.shared import Emu, Inches, Pt
from fastcore.basics import patch, store_attr
from mistletoe import Document as MdDocument
from mistletoe.span_token import SpanToken, RawText, Strong, Emphasis
from mistletoe import span_token

import re

In [ ]:
#| export
_docx_tracking = None

def set_tracking(author=None):
    "Set author name to enable tracked changes, or None to disable"
    global _docx_tracking
    _docx_tracking = author

class Revision:
    "Tracked change metadata: typ ('ins'/'del'), author, date"
    def __init__(self, typ, author, date): store_attr()
    def __repr__(self): return f'Revision(typ={self.typ!r}, author={self.author!r}, date={self.date!r})'

@patch(as_prop=True)
def revision(self:Run):
    "Tracked change metadata from parent w:ins/w:del, or None"
    p = self._r.getparent()
    tag = p.tag.split('}')[-1] if p is not None else None
    if tag not in ('ins', 'del'): return None
    return Revision(typ=tag, author=p.get(qn('w:author')), date=p.get(qn('w:date')))

@patch(as_prop=True)
def text(self:Run):
    "Get run text from w:t or w:delText"
    return ''.join(el.text for el in self._r
                   if el.tag.split('}')[-1] in ('t', 'delText') and el.text)

@patch(set_prop=True)
def text(self:Run, value): self._r.text = value

@patch
def _repr_markdown_(self:Run):
    "Render run as markdown with formatting and revision spans"
    t = self.text
    if not t: return ''
    b, i, u = self.font.bold, self.font.italic, self.font.underline
    rev = getattr(self, 'revision', None)
    if not b and not i and not u and not rev: return t
    lead, trail = len(t)-len(t.lstrip()), len(t)-len(t.rstrip())
    inner = t.strip()
    if not inner: return t
    pre, suf = t[:lead], (t[len(t)-trail:] if trail else '')
    if   b and i: inner = f'***{inner}***'
    elif b:       inner = f'**{inner}**'
    elif i:       inner = f'*{inner}*'
    if u: inner = f'<u>{inner}</u>'
    res = f'{pre}{inner}{suf}'
    if rev:
        cls = 'deletion' if rev.typ == 'del' else 'insertion'
        res = f'<span class="{cls}">{res}</span>'
    return res

In [ ]:
doc = DocxDocument()
p = doc.add_paragraph()
run = p.add_run('Hello world')
run

<div class="prose">

Hello world

</div>

In [ ]:
run.font.bold = True
run

<div class="prose">

**Hello world**

</div>

In [ ]:
#| export
@patch(as_prop=True)
def runs(self:Paragraph):
    "All runs in document order, including those inside w:ins and w:del"
    runs = []
    for el in self._p:
        tag = el.tag.split('}')[-1]
        if tag == 'r': runs.append(Run(el, self))
        elif tag in ('ins', 'del'):
            for r_el in el.findall(qn('w:r')): runs.append(Run(r_el, self))
    return runs

def _rev_el(typ, author):
    "Create a w:ins or w:del element with author and timestamp"
    el = OxmlElement(f'w:{typ}')
    el.set(qn('w:id'), str(id(el)))
    el.set(qn('w:author'), author)
    el.set(qn('w:date'), datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'))
    return el

@patch
def add_run(self:Paragraph, text=None, style=None):
    "Add a run, wrapping in w:ins if tracking is set"
    run = self._orig_add_run(text, style)
    if _docx_tracking:
        el = _rev_el('ins', _docx_tracking)
        run._r.addprevious(el)
        el.append(run._r)
    return run

@patch
def _repr_markdown_(self:Paragraph):
    "Render paragraph as markdown with style tag and optional indent"
    list_style = ''
    if self.style.name == 'List Number': list_style = '1. '
    elif self.style.name == 'List Bullet': list_style = '- '
    runs = self.runs
    if not any(r.text.strip() for r in runs): return ''
    return list_style+''.join(r._repr_markdown_() for r in runs)

In [ ]:
set_tracking('Author Name')
run = p.add_run('. A new run!')
run.italic = True
p

<div class="prose">

**Hello world**<span class="insertion">*. A new run!*</span>

</div>

In [ ]:
run.revision

Revision(typ='ins', author='Author Name', date='2026-05-06T01:37:45Z')

In [ ]:
#| export
class Underline(SpanToken):
    pattern = re.compile(r'<u>(.+?)</u>')
    parse_inner = True
    parse_group = 1

span_token.add_token(Underline)

def _walk(node, para, bold=False, italic=False, underline=False):
    "Walk mistletoe AST, adding runs to paragraph with formatting"
    if isinstance(node, RawText):
        run = para.add_run(node.content)
        if bold: run.font.bold = True
        if italic: run.font.italic = True
        if underline: run.font.underline = True
        return
    for child in node.children:
        _walk(child, para,
              bold or isinstance(node, Strong),
              italic or isinstance(node, Emphasis),
              underline or isinstance(node, Underline))

@patch
def add_md(self:Paragraph, text):
    "Parse markdown text and add as formatted runs"
    doc = MdDocument(text)
    for block in doc.children:
        for child in block.children: _walk(child, self)
    return self

In [ ]:
set_tracking(None)
new_p = doc.add_paragraph()
new_p = new_p.add_md('*italics*, **bold**, <u>underlying</u>, and <u>***all of them***</u>')
new_p

<div class="prose">

*italics*, **bold**, <u>underlying</u>, and <u>***all of them***</u>

</div>

In [ ]:
#| export
@patch
def delete(self:Run):
    "Delete run, or mark as tracked deletion if tracking is set"
    if _docx_tracking:
        el = _rev_el('del', _docx_tracking)
        for t in self._r.findall(qn('w:t')): t.tag = qn('w:delText')
        self._r.addprevious(el)
        el.append(self._r)
    else: self._r.getparent().remove(self._r)

In [ ]:
new_r = new_p.add_run('This is going to be deleted.')
new_p

<div class="prose">

*italics*, **bold**, <u>underlying</u>, and <u>***all of them***</u>This is going to be deleted.

</div>

In [ ]:
set_tracking('Author Name')
new_r.delete()
new_p

<div class="prose">

*italics*, **bold**, <u>underlying</u>, and <u>***all of them***</u><span class="deletion">This is going to be deleted.</span>

</div>

In [ ]:
new_r.revision

Revision(typ='del', author='Author Name', date='2026-05-06T01:37:45Z')

In [ ]:
#| export
@patch
def add_row(self:Table, *cells):
    row = self._orig_add_row()
    for cell, val in zip(row.cells, cells): cell.paragraphs[0].add_md(str(val))
    return row

def _content_width(part):
    "Content width (page minus margins) from the document's last section"
    doc = part.document
    section = doc.sections[-1]
    pw = section.page_width  or Inches(8.5)
    lm = section.left_margin or Inches(1)
    rm = section.right_margin or Inches(1)
    return Emu(pw - lm - rm)

@patch
def insert_table_after(self:Paragraph, rows, cols):
    "Insert table with `rows` rows and `cols` cols after self, return Table"
    tbl_el = CT_Tbl.new_tbl(rows, cols, _content_width(self.part))
    self._p.addnext(tbl_el)
    return Table(tbl_el, self._parent)

@patch
def insert_table_before(self:Paragraph, rows, cols):
    "Insert table with `rows` rows and `cols` cols before self, return Table"
    tbl_el = CT_Tbl.new_tbl(rows, cols, _content_width(self.part))
    self._p.addprevious(tbl_el)
    return Table(tbl_el, self._parent)

@patch
def insert_after(self:Table, rows, cols):
    "Insert table with `rows` rows and `cols` cols after self, return Table"
    tbl_el = CT_Tbl.new_tbl(rows, cols, _content_width(self.part))
    self._tbl.addnext(tbl_el)
    return Table(tbl_el, self._parent)

@patch
def insert_before(self:Table, rows, cols):
    "Insert table with `rows` rows and `cols` cols before self, return Table"
    tbl_el = CT_Tbl.new_tbl(rows, cols, _content_width(self.part))
    self._tbl.addprevious(tbl_el)
    return Table(tbl_el, self._parent)

def _cell_md(cell):
    "Render all runs in a table cell as inline markdown"
    parts = [r._repr_markdown_() for p in cell.paragraphs for r in p.runs]
    return ' '.join(parts).strip().replace('\n', ' ')

@patch
def _repr_markdown_(self:Table):
    "Render table as markdown with style tag"
    rows = [[_cell_md(c) for c in row.cells] for row in self.rows]
    if not rows: return ''
    hdr = '| ' + ' | '.join(rows[0]) + ' |'
    sep = '| ' + ' | '.join('---' for _ in rows[0]) + ' |'
    body = '\n'.join('| ' + ' | '.join(r) + ' |' for r in rows[1:])
    return f'{hdr}\n{sep}\n{body}'

In [ ]:
tbl = doc.add_table(rows=0, cols=3)
for i in range(3):
    tbl.add_row(f'R{i}C0', f'R{i}C1', f'R{i}C2')

tbl

<div class="prose">

| <span class="insertion">R0C0</span> | <span class="insertion">R0C1</span> | <span class="insertion">R0C2</span> |
| --- | --- | --- |
| <span class="insertion">R1C0</span> | <span class="insertion">R1C1</span> | <span class="insertion">R1C2</span> |
| <span class="insertion">R2C0</span> | <span class="insertion">R2C1</span> | <span class="insertion">R2C2</span> |

</div>

In [ ]:
#| export
@patch
def _repr_markdown_(self:Document):
    "Render full document as markdown"
    res = []
    for el in self.element.body:
        tag = el.tag.split('}')[-1]
        if   tag == 'p':   md = Paragraph(el, self)._repr_markdown_()
        elif tag == 'tbl': md = Table(el, self)._repr_markdown_()
        else: continue
        if md: res.append(md)
    return '\n\n'.join(res)

In [ ]:
doc

<div class="prose">

**Hello world**<span class="insertion">*. A new run!*</span>

*italics*, **bold**, <u>underlying</u>, and <u>***all of them***</u><span class="deletion">This is going to be deleted.</span>

| <span class="insertion">R0C0</span> | <span class="insertion">R0C1</span> | <span class="insertion">R0C2</span> |
| --- | --- | --- |
| <span class="insertion">R1C0</span> | <span class="insertion">R1C1</span> | <span class="insertion">R1C2</span> |
| <span class="insertion">R2C0</span> | <span class="insertion">R2C1</span> | <span class="insertion">R2C2</span> |

</div>

In [ ]:
tbl2 = new_p.insert_table_before(0, 2)
tbl2.add_row('Col A', 'Col B')
doc

<div class="prose">

**Hello world**<span class="insertion">*. A new run!*</span>

| <span class="insertion">Col A</span> | <span class="insertion">Col B</span> |
| --- | --- |


*italics*, **bold**, <u>underlying</u>, and <u>***all of them***</u><span class="deletion">This is going to be deleted.</span>

| <span class="insertion">R0C0</span> | <span class="insertion">R0C1</span> | <span class="insertion">R0C2</span> |
| --- | --- | --- |
| <span class="insertion">R1C0</span> | <span class="insertion">R1C1</span> | <span class="insertion">R1C2</span> |
| <span class="insertion">R2C0</span> | <span class="insertion">R2C1</span> | <span class="insertion">R2C2</span> |

</div>

In [ ]:
#| export
@patch
def delete(self:Paragraph):
    "Delete paragraph, or mark all runs as tracked deletions"
    if _docx_tracking:
        for r in self.runs:
            if (rev:=r.revision) and rev.typ=='del': continue
            r.delete()
    else: self._p.getparent().remove(self._p)

In [ ]:
new_p.delete()
doc

<div class="prose">

**Hello world**<span class="insertion">*. A new run!*</span>

| <span class="insertion">Col A</span> | <span class="insertion">Col B</span> |
| --- | --- |


<span class="deletion">*italics*</span><span class="deletion">, </span><span class="deletion">**bold**</span><span class="deletion">, </span><span class="deletion"><u>underlying</u></span><span class="deletion">, and </span><span class="deletion"><u>***all of them***</u></span><span class="deletion">This is going to be deleted.</span>

| <span class="insertion">R0C0</span> | <span class="insertion">R0C1</span> | <span class="insertion">R0C2</span> |
| --- | --- | --- |
| <span class="insertion">R1C0</span> | <span class="insertion">R1C1</span> | <span class="insertion">R1C2</span> |
| <span class="insertion">R2C0</span> | <span class="insertion">R2C1</span> | <span class="insertion">R2C2</span> |

</div>

In [ ]:
set_tracking(None)
for i in range(1, 4): doc.add_paragraph(f'Item {i}', style='List Number')
paras = doc.paragraphs
doc

<div class="prose">

**Hello world**<span class="insertion">*. A new run!*</span>

| <span class="insertion">Col A</span> | <span class="insertion">Col B</span> |
| --- | --- |


<span class="deletion">*italics*</span><span class="deletion">, </span><span class="deletion">**bold**</span><span class="deletion">, </span><span class="deletion"><u>underlying</u></span><span class="deletion">, and </span><span class="deletion"><u>***all of them***</u></span><span class="deletion">This is going to be deleted.</span>

| <span class="insertion">R0C0</span> | <span class="insertion">R0C1</span> | <span class="insertion">R0C2</span> |
| --- | --- | --- |
| <span class="insertion">R1C0</span> | <span class="insertion">R1C1</span> | <span class="insertion">R1C2</span> |
| <span class="insertion">R2C0</span> | <span class="insertion">R2C1</span> | <span class="insertion">R2C2</span> |

1. Item 1

1. Item 2

1. Item 3

</div>

In [ ]:
#| export
def _copy_para_fmt(src, dst):
    "Copy style and numbering from src paragraph to dst"
    dst.style = src.style
    src_pPr = src._p.find(qn('w:pPr'))
    if src_pPr is not None:
        src_num = src_pPr.find(qn('w:numPr'))
        if src_num is not None:
            dst_pPr = dst._p.get_or_add_pPr()
            dst_pPr.append(deepcopy(src_num))

@patch
def insert_before(self:Paragraph, text, ref=None):
    "Insert paragraph before self with markdown text, copying format from ref (default: self)"
    ref = ref or self
    new_p_el = self._p.add_p_before()
    new_p = Paragraph(new_p_el, self._parent)
    _copy_para_fmt(ref, new_p)
    new_p.add_md(text)
    return new_p

@patch
def insert_after(self:Paragraph, text, ref=None):
    "Insert paragraph after self with markdown text, copying format from ref (default: self)"
    ref = ref or self
    new_p_el = self._p.add_p_before()
    self._p.addnext(new_p_el)
    new_p = Paragraph(new_p_el, self._parent)
    _copy_para_fmt(ref, new_p)
    new_p.add_md(text)
    return new_p

In [ ]:
set_tracking('Author Name')
paras[-3].insert_after('Inserting after 1!')
doc

<div class="prose">

**Hello world**<span class="insertion">*. A new run!*</span>

| <span class="insertion">Col A</span> | <span class="insertion">Col B</span> |
| --- | --- |


<span class="deletion">*italics*</span><span class="deletion">, </span><span class="deletion">**bold**</span><span class="deletion">, </span><span class="deletion"><u>underlying</u></span><span class="deletion">, and </span><span class="deletion"><u>***all of them***</u></span><span class="deletion">This is going to be deleted.</span>

| <span class="insertion">R0C0</span> | <span class="insertion">R0C1</span> | <span class="insertion">R0C2</span> |
| --- | --- | --- |
| <span class="insertion">R1C0</span> | <span class="insertion">R1C1</span> | <span class="insertion">R1C2</span> |
| <span class="insertion">R2C0</span> | <span class="insertion">R2C1</span> | <span class="insertion">R2C2</span> |

1. Item 1

1. <span class="insertion">Inserting after 1!</span>

1. Item 2

1. Item 3

</div>

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()